# 03 - Renewable Generation Forecasting
## Leakage-Safe Feature Engineering, Baseline Evaluation, and ML Models

This notebook trains day-ahead hourly solar forecasting models using strictly historical information (lag >= 24h), solar geometry clear-sky proxies, and evaluates them against Persistence and Seasonal Naive baselines.


In [1]:
import sys
sys.path.insert(0, '..')

from src.preprocessing import get_clean_data
from src.features import build_forecasting_features, verify_feature_leakage
from src.forecasting import train_and_evaluate_models
import pandas as pd

clean_df = get_clean_data()
print("Running anti-leakage verification test...")
verify_feature_leakage(clean_df)


[2026-10-25 14:02:42] [INFO] [src.features]: Leakage verification test PASSED. Features are strictly non-anticipative.


Running anti-leakage verification test...


True

### 1. Train Models & Evaluate on Chronological Holdout Set (2020)

In [2]:
preds, metrics, leaderboard = train_and_evaluate_models(clean_df, force_retrain=False)
print("=== Model Leaderboard (Sorted by MAE) ===")
display(leaderboard)


[2026-10-25 14:02:42] [INFO] [src.forecasting]: Chronological split sizes: Train=34,729 (2015-01-14 to 2018-12-31), Val=8,760 (2019-01-01 to 2019-12-31), Test=6,576 (2020-01-01 to 2020-09-30)


[2026-10-25 14:02:42] [INFO] [src.forecasting]: Model feature columns (24): ['capacity_proxy_mw', 'solar_elevation_deg', 'clearsky_proxy', 'is_daylight', 'hour', 'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'month', 'doy_sin', 'doy_cos', 'target_lag_last_available_norm', 'target_lag_48h_norm', 'target_lag_168h_norm', 'target_rolling_mean_24h_norm', 'target_rolling_std_24h_norm', 'target_rolling_max_24h_norm', 'target_rolling_mean_168h_norm', 'price_day_ahead_eur_mwh', 'price_daily_mean_eur_mwh', 'source_load_forecast_mw', 'wind_lag_safe', 'load_lag_safe']


/home/asus/anaconda3/envs/renewable-ei/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
[2026-10-25 14:02:43] [INFO] [src.forecasting]: Training LightGBM quantile regressors (P10, P50, P90)...


[2026-10-25 14:02:44] [INFO] [src.forecasting]: Model training and evaluation successfully completed!


[2026-10-25 14:02:44] [INFO] [src.forecasting]: 
                   model                       type  mae_all  rmse_all  nmae_all_pct  mae_daylight  skill_score  train_time_sec  inference_time_ms  size_kb
0          Random Forest                Ensemble ML   39.578    74.413          5.45        70.832       0.0483           0.954              22.32  13285.6
1               LightGBM          Gradient Boosting   39.710    74.388          5.46        71.070       0.0451           0.175               6.20    226.8
2       Ridge Regression                  Linear ML   39.925    74.362          5.49        71.456       0.0400           0.005               0.50      0.7
3     Last-Available-Day  Baseline (Auction Cutoff)   41.587    88.547          5.72        74.444       0.0000           0.000               0.00      0.0
4   Seasonal Naive (48h)         Baseline (Lag 48h)   45.203    94.722          6.22        80.943      -0.0870           0.000               0.00      0.0
5  Seasonal Nai

=== Model Leaderboard (Sorted by MAE) ===


,model,type,mae_all,rmse_all,nmae_all_pct,mae_daylight,skill_score,train_time_sec,inference_time_ms,size_kb
0,Random Forest,Ensemble ML,39.578,74.413,5.45,70.832,0.0483,0.954,22.32,13285.6
1,LightGBM,Gradient Boosting,39.710,74.388,5.46,71.070,0.0451,0.175,6.20,226.8
2,Ridge Regression,Linear ML,39.925,74.362,5.49,71.456,0.0400,0.005,0.50,0.7
3,Last-Available-Day,Baseline (Auction Cutoff),41.587,88.547,5.72,74.444,0.0000,0.000,0.00,0.0
4,Seasonal Naive (48h),Baseline (Lag 48h),45.203,94.722,6.22,80.943,-0.0870,0.000,0.00,0.0
5,Seasonal Naive (168h),Baseline (Lag 168h),51.547,103.379,7.09,92.348,-0.2395,0.000,0.00,0.0


### 2. Conformal Prediction Intervals & Empirical Coverage

In [3]:
print("90% Conformal Prediction Interval Metrics:")
print(metrics.get("Conformal_90_Coverage"))
print("80% Conformal Prediction Interval Metrics:")
print(metrics.get("Conformal_80_Coverage"))


90% Conformal Prediction Interval Metrics:
{'overall_coverage_pct': 73.91, 'overall_mean_width_mw': 102.28, 'daylight_coverage_pct': 70.95, 'daylight_mean_width_mw': 183.86, 'night_coverage_pct': 77.62, 'night_mean_width_mw': 0.0, 'seasonal': {'Winter': {'coverage_pct': 88.26, 'mean_width_mw': 45.71, 'count_hours': 1440}, 'Spring': {'coverage_pct': 67.53, 'mean_width_mw': 101.32, 'count_hours': 2208}, 'Summer': {'coverage_pct': 69.76, 'mean_width_mw': 139.02, 'count_hours': 2179}, 'Autumn': {'coverage_pct': 77.36, 'mean_width_mw': 107.15, 'count_hours': 720}}}
80% Conformal Prediction Interval Metrics:
{'overall_coverage_pct': 69.45, 'overall_mean_width_mw': 79.66, 'daylight_coverage_pct': 62.93, 'daylight_mean_width_mw': 143.21, 'night_coverage_pct': 77.62, 'night_mean_width_mw': 0.0, 'seasonal': {'Winter': {'coverage_pct': 87.08, 'mean_width_mw': 40.81, 'count_hours': 1440}, 'Spring': {'coverage_pct': 63.18, 'mean_width_mw': 79.53, 'count_hours': 2208}, 'Summer': {'coverage_pct': 63.